# 03 · Statistics — NYC 311 Resolution Time

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
from pathlib import Path
import os

project = Path(os.environ.get("URBANPULSE_ROOT", Path.cwd())).resolve()
if not (project / "data").is_dir() and (project.parent / "data").is_dir():
    project = project.parent
if not (project / "data").is_dir():
    raise FileNotFoundError("Run from the repository root or set URBANPULSE_ROOT.")


data_path = project / "data"
stats_path = data_path / "statistics"
stats_path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path / "cleaned" / "nyc_cleaned.csv", low_memory=False)
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

# Valid durations only: numeric, finite and non-negative
df["resolution_hours"] = pd.to_numeric(df["resolution_hours"], errors="coerce")
valid_resolution = df.loc[np.isfinite(df["resolution_hours"]) & (df["resolution_hours"] >= 0),
                          "resolution_hours"]
if valid_resolution.empty:
    raise ValueError("No valid resolution records available.")

Rows: 100000 | Columns: 41


## 1. Central tendency and spread

For skewed resolution times the **median is reported alongside the mean**. Spread is calculated for `resolution_hours` and for the coordinate fields.

In [2]:
def describe_numeric(s, with_mode=False):
    s = pd.to_numeric(s, errors="coerce")
    s = s[np.isfinite(s)]
    q1, med, q3 = s.quantile([0.25, 0.50, 0.75])
    out = {"Count": len(s), "Mean": s.mean(), "Median": med}
    if with_mode:
        out["Mode"] = s.mode().iloc[0] if not s.mode().empty else np.nan
    out.update({"Minimum": s.min(), "Maximum": s.max(), "Range": s.max() - s.min(),
                "Q1": q1, "Q3": q3, "IQR": q3 - q1,
                "Variance (sample)": s.var(ddof=1), "Std deviation (sample)": s.std(ddof=1),
                "Skewness": s.skew()})
    return out

v = valid_resolution
res = describe_numeric(v, with_mode=True)
mean, median, q1, q3, iqr, skewness = (res[k] for k in ["Mean", "Median", "Q1", "Q3", "IQR", "Skewness"])

numeric_summary = pd.DataFrame({
    "resolution_hours": res,
    "latitude": describe_numeric(df["latitude"]),
    "longitude": describe_numeric(df["longitude"]),
})
display(numeric_summary.round(2))

if pd.isna(skewness):
    print("Skewness cannot be determined.")
elif skewness > 1:
    print(f"Highly right-skewed (mean {mean:.2f} h > median {median:.2f} h): long cases pull the mean up; report the median alongside it.")
elif skewness < -1:
    print(f"Highly left-skewed (mean {mean:.2f} h, median {median:.2f} h): report the median alongside the mean.")
else:
    print(f"Not highly skewed (±1 threshold; mean {mean:.2f} h, median {median:.2f} h): both are useful.")

,resolution_hours,latitude,longitude
Count,69263.00,98415.00,98415.00
Mean,16.32,40.73,-73.93
Median,2.64,40.73,-73.93
Mode,0.00,NaN,NaN
Minimum,0.00,40.50,-74.25
Maximum,199.47,40.91,-73.70
Range,199.47,0.41,0.55
Q1,0.74,40.67,-73.97
Q3,18.05,40.81,-73.89
IQR,17.31,0.14,0.08


Highly right-skewed (mean 16.32 h > median 2.64 h): long cases pull the mean up; report the median alongside it.


## 2. Most frequent values (categorical and time fields)

In [3]:
def mode_summary(columns):
    rows = []
    for col in columns:
        if col not in df.columns:
            continue
        s = df[col].dropna()
        s = s[s.astype(str).str.strip() != ""]
        if s.empty:
            continue
        counts = s.value_counts()
        rows.append({"Field": col, "Most Frequent Value": counts.index[0],
                     "Frequency": counts.iloc[0],
                     "Percentage": round(counts.iloc[0] / len(s) * 100, 2)})
    return pd.DataFrame(rows)

categorical_summary = mode_summary(["agency", "agency_name", "complaint_type", "descriptor",
                                    "borough", "status", "location_type", "city", "incident_zip"])
temporal_summary = mode_summary(["created_year", "created_month", "created_weekday", "created_hour"])
display(categorical_summary)
display(temporal_summary)

,Field,Most Frequent Value,Frequency,Percentage
0,agency,NYPD,41731,41.73
1,agency_name,NEW YORK CITY POLICE DEPARTMENT,41731,41.73
2,complaint_type,ILLEGAL PARKING,16398,16.40
3,descriptor,ENTIRE BUILDING,8564,8.69
4,borough,BROOKLYN,32779,32.78
5,status,CLOSED,66944,66.94
6,location_type,STREET/SIDEWALK,31268,33.72
7,city,BROOKLYN,31241,33.10
8,incident_zip,11226.0,1971,1.99


,Field,Most Frequent Value,Frequency,Percentage
0,created_year,2026,100000,100.00
1,created_month,10,100000,100.00
2,created_weekday,Thursday,22084,22.08
3,created_hour,11,6113,6.11


## 3. Resolution time by group

In [4]:
df.loc[~np.isfinite(df["resolution_hours"]) | (df["resolution_hours"] < 0), "resolution_hours"] = np.nan

def group_stats(col, sort_by):
    return (df.groupby(col)["resolution_hours"].agg(Count="count", Mean="mean", Median="median")
              .round(2).sort_values(sort_by, ascending=False))

borough_stats   = group_stats("borough", "Median")
agency_stats    = group_stats("agency", "Median")
complaint_stats = group_stats("complaint_type", "Count")

display(borough_stats)
display(agency_stats)
display(complaint_stats.head(20))

,Count,Mean,Median
borough,,,
UNSPECIFIED,46,30.19,19.10
BRONX,13019,19.43,3.94
STATEN ISLAND,2428,16.73,3.42
MANHATTAN,14403,17.87,2.56
BROOKLYN,22911,16.09,2.38
QUEENS,16456,12.73,2.26


,Count,Mean,Median
agency,,,
OTI,1,119.83,119.83
TLC,126,73.23,81.18
DOHMH,593,63.76,50.29
HPD,10266,46.53,36.65
DSNY,5355,38.45,28.84
DCWP,184,36.48,25.46
DPR,469,40.11,23.35
OOS,16,50.41,19.52
DEP,3084,33.16,19.35


,Count,Mean,Median
complaint_type,,,
ILLEGAL PARKING,16207,2.67,1.43
NOISE - RESIDENTIAL,8847,2.41,1.20
HEAT/HOT WATER,7054,36.76,32.71
BLOCKED DRIVEWAY,4846,2.73,1.52
NOISE - STREET/SIDEWALK,3922,2.00,1.01
ABANDONED VEHICLE,1800,3.77,2.13
NOISE - COMMERCIAL,1644,1.78,0.97
STREET CONDITION,1499,39.29,22.10
DIRTY CONDITION,1318,49.64,46.51


## 4. Outliers — Tukey's 1.5 × IQR rule

Outliers are **flagged, not removed**; long service cases may be legitimate.

In [5]:
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_mask = (v < lower) | (v > upper)
n_out = int(outlier_mask.sum())

outlier_summary = pd.DataFrame({
    "Measure": ["Q1 (hours)", "Q3 (hours)", "IQR (hours)", "Lower fence (hours)",
                "Upper fence (hours)", "Flagged outlier count", "Flagged outlier percentage (%)"],
    "Value": [q1, q3, iqr, lower, upper, n_out, 100 * n_out / len(v)],
})
display(outlier_summary.round(2))

# Flag in the DataFrame (all records kept)
df["resolution_outlier_iqr"] = pd.Series(pd.NA, index=df.index, dtype="boolean")
df.loc[v.index, "resolution_outlier_iqr"] = outlier_mask

preview_cols = [c for c in ["unique_key", "agency", "complaint_type", "borough",
                            "resolution_hours", "resolution_outlier_iqr"] if c in df.columns]
flagged = df.loc[df["resolution_outlier_iqr"].fillna(False), preview_cols]
display(flagged.head(20))
print(f"{n_out:,} of {len(v):,} valid durations ({100 * n_out / len(v):.2f}%) lie outside "
      f"[{lower:.2f}, {upper:.2f}] hours. These are investigation prompts, not deletions.")

,Measure,Value
0,Q1 (hours),0.74
1,Q3 (hours),18.05
2,IQR (hours),17.31
3,Lower fence (hours),-25.23
4,Upper fence (hours),44.02
5,Flagged outlier count,9250.00
6,Flagged outlier percentage (%),13.35


,unique_key,agency,complaint_type,borough,resolution_hours,resolution_outlier_iqr
23232,70853198,DOT,STREET CONDITION,QUEENS,46.083056,True
23234,70850274,DOT,STREET CONDITION,QUEENS,44.763056,True
23236,70853197,DOT,STREET CONDITION,QUEENS,44.195000,True
23242,70857722,DOT,STREET CONDITION,BRONX,45.883056,True
23295,70857731,DOT,STREET CONDITION,MANHATTAN,45.569444,True
23312,70853190,DOT,STREET CONDITION,MANHATTAN,44.763611,True
23509,70826130,DEP,NOISE,MANHATTAN,46.003333,True
24294,70831271,DEP,NOISE,QUEENS,44.239722,True
24704,70827869,DEP,NOISE,MANHATTAN,48.730000,True
24958,70830036,DOHMH,INDOOR AIR QUALITY,QUEENS,44.742500,True


9,250 of 69,263 valid durations (13.35%) lie outside [-25.23, 44.02] hours. These are investigation prompts, not deletions.


## 5. Distribution — histogram and boxplot

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.hist(v, bins=50, color="skyblue", edgecolor="white")
ax1.axvline(mean, color="red", linestyle="--", label=f"Mean = {mean:.2f} h")
ax1.axvline(median, color="green", label=f"Median = {median:.2f} h")
ax1.set(title="NYC 311 resolution time distribution", xlabel="Resolution hours",
        ylabel="Number of requests")
ax1.legend()

ax2.boxplot(v, vert=False, whis=1.5)
ax2.axvline(upper, linestyle="--", label=f"Upper fence = {upper:.2f} h")
ax2.set(title="Resolution time boxplot", xlabel="Resolution hours")
ax2.legend()

plt.tight_layout()
plt.savefig(stats_path / "resolution_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Valid closed-case durations: {len(v):,} | Mean {mean:.2f} h | Median {median:.2f} h | Skewness {skewness:.3f}")
print("Inspect the observed distribution before interpreting it. Flagged durations are retained.")

C:\Users\HSP\AppData\Local\Temp\ipykernel_19356\340016460.py:10: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax2.boxplot(v, vert=False, whis=1.5)


Valid closed-case durations: 69,263 | Mean 16.32 h | Median 2.64 h | Skewness 2.727
Inspect the observed distribution before interpreting it. Flagged durations are retained.


C:\Users\HSP\AppData\Local\Temp\ipykernel_19356\340016460.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Save results

In [7]:
flat_tables = {"resolution_iqr_outlier_summary": outlier_summary,
               "resolution_iqr_flagged_cases": flagged,
               "categorical_modes": categorical_summary,
               "temporal_modes": temporal_summary}
group_tables = {"numeric_summary": numeric_summary,
                "borough_central_tendency": borough_stats,
                "agency_central_tendency": agency_stats,
                "complaint_central_tendency": complaint_stats}

for name, table in flat_tables.items():
    table.to_csv(stats_path / f"{name}.csv", index=False)
for name, table in group_tables.items():
    table.to_csv(stats_path / f"{name}.csv")          # keep the group label index

print("Saved", len(flat_tables) + len(group_tables) + 1, "files to:", stats_path)

Saved 9 files to: D:\Coding\UrbanPulse\data\statistics
